# 🎙️ OmniVoice Studio

Estúdio de voz e audiobook para usar **pelo celular**.

## Como usar

1. Menu **Ambiente de execução → Alterar o tipo de ambiente → GPU (T4)**
2. Menu **Ambiente de execução → Executar tudo**
3. Autorize o Google Drive quando for pedido (opcional, mas recomendado)
4. Espere aparecer o link `https://....trycloudflare.com`
5. Abra esse link no navegador do celular

> A última célula precisa continuar rodando enquanto você usa o Studio.
> Sem GPU o Studio abre em modo simulador: a interface toda funciona,
> mas o áudio não é fala de verdade.


## 1. Preparar o ambiente


In [ ]:
#@title Preparar { display-mode: "form" }
#@markdown Deixe o Drive ligado para que vozes, textos e áudios
#@markdown sobrevivam ao fim da sessão do Colab.
usar_google_drive = True  #@param {type:"boolean"}
#@markdown Deixe a branch vazia para usar a padrão do repositório.
branch = ""  #@param {type:"string"}

import importlib
import pathlib
import shutil
import subprocess
import sys

REPO = "https://github.com/werikvinicios-dev/ominivoiceapp.git"
DEST = pathlib.Path("/content/ominivoiceapp")

# Clone sempre do zero. Reaproveitar a pasta deixava código antigo para trás
# quando o git clone falhava em silêncio por ela já existir.
shutil.rmtree(DEST, ignore_errors=True)
clone = ["git", "clone", "--depth", "1"]
if branch:
    clone += ["--branch", branch]
subprocess.run([*clone, REPO, str(DEST)], check=True)

# Descarta o módulo que o kernel já tenha em memória: sem isto, "import colab"
# reusaria a versão antiga e nenhuma atualização teria efeito.
sys.path.insert(0, str(DEST / "scripts"))
for nome in [m for m in sys.modules if m == "colab" or m.startswith("colab.")]:
    del sys.modules[nome]
import colab

importlib.reload(colab)

commit = subprocess.run(
    ["git", "-C", str(DEST), "log", "-1", "--format=%h %s"],
    capture_output=True, text=True, check=False,
).stdout.strip()
print(f"Versão do Studio: {commit}")
print(f"Lançador: {colab.__file__}\n")

config = colab.setup(use_drive=usar_google_drive, branch=branch)
config

## 2. Abrir o OmniVoice Studio

Quando o link aparecer, abra-o no celular. **Deixe esta célula rodando.**


In [ ]:
#@title Abrir OmniVoice Studio { display-mode: "form" }
sessao = colab.launch(config)

if sessao.get("url"):
    from IPython.display import HTML, display

    display(
        HTML(
            f'<a href="{sessao["url"]}" target="_blank" '
            'style="display:inline-block;padding:16px 28px;margin:12px 0;'
            'background:#6d8dff;color:#0b0d12;border-radius:12px;'
            'font:600 17px system-ui;text-decoration:none">'
            '🎙️ ABRIR OMNIVOICE STUDIO</a>'
        )
    )

# Mantém a célula viva e reabre o túnel se ele cair (erro 1033).
colab.keep_alive(sessao)


## 3. Diagnóstico (só se algo der errado)

Pare a célula 2, rode esta, e me mostre a saída.


In [ ]:
#@title Diagnosticar { display-mode: "form" }
colab.diagnose(sessao)


## Problemas?

| Sintoma | O que fazer |
|---|---|
| Diz “modo simulador” | Troque o ambiente para GPU e execute tudo de novo. |
| O link não apareceu | Rode a célula 2 novamente; o túnel às vezes falha na primeira tentativa. |
| Erro **1033** | **Confira primeiro se o link é o desta execução** — os endereços mudam a cada vez, e um link antigo sempre dá 1033. Se for o link atual, use o **link do Colab** (o segundo da lista), que não depende de túnel. Depois rode a célula de diagnóstico. |
| O microfone não grava | Use o link `https://`, não um IP. Se ainda assim não gravar, envie um arquivo de áudio. |
| Quero ver o erro do servidor | `!tail -50 /content/omnivoice-studio.log` |
| Quero ver o erro do túnel | `!tail -30 /content/cloudflared.log` |

Os dados persistentes ficam em **`Meu Drive/OmniVoiceStudio`**:
banco (`studio.db`), vozes (`voices/`) e áudios finais (`audio/`).
